# Imports

In [ ]:
# Cuda
import torch

# Starter Code
# # Provided 
# import torch
from torchvision.transforms import v2
from PIL import Image
import os
import json
from typing import Callable
# # Added
from torch.utils.data import DataLoader

# Text Tokenization
from collections import Counter
import re

# Challenge #2
# from collections import Counter
# import json

# Challenge #3
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# Challenge #4
# from collections import Counter
# import json

In [ ]:
print(f"Is CUDA available? {torch.cuda.is_available()}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load The Data

In [ ]:
# Starter Code Provided on Canvas as VizWiz_Loader.py
class VizWizLoader(torch.utils.data.Dataset):
    def __init__(self, strFolder: str, strAnnotationPath: str, fDataPercentage: float = 1.0,
                 tTransform: Callable[[torch.tensor], torch.tensor] = None) -> None:
        '''
        strFolder: path to unzip'd folder of VizWiz images
        strLabelPath: path to .json file containing the annotations
        fDataPercentage: percentage of available samples to use. Must be normalized between 0.0 and 1.0. Default: 1.0
        tTransform: optional place to connect PyTorch image transformations. Default: converts images to 3x224x224 tensors
        For the train and val splits, returns tuples of the form:
            (image, question text, binary label, answer texts)
        Otherwise, returns tuples of the form:
            (image, question text)
        '''
        self.strFolder = strFolder
        if self.strFolder[-1] != "/": self.strFolder += "/"
        self.tTransform = tTransform
        vecPaths = os.listdir(self.strFolder)
        self.strPrefix = vecPaths[0].split("_")[1]

        with open(strAnnotationPath, "r") as f:
            self.vecAnnos = json.load(f)

        self.iN = int(fDataPercentage * len(self.vecAnnos))

        if self.tTransform is None:
            self.tTransform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale = True), v2.RandomCrop(224)])
            self.tTransformUndersized = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale = True), v2.Resize((224, 224))])

        return

    def __len__(self) -> int: return self.iN

    def __getitem__(self, idx: int) -> tuple:
        if idx > self.iN:
            print(f"Error! Tried to access index {idx} but only {self.iN} samples are available")
            return None

        strPath = self.strFolder + self.vecAnnos[idx]["image"]
        imX = Image.open(strPath)
        w, h = imX.size
        if w >= 224 and h >= 224: tX = self.tTransform(imX)
        else: tX = self.tTransformUndersized(imX)

        if self.strPrefix == "test":
            return tX, self.vecAnnos[idx]["question"]
        else:
            return tX, self.vecAnnos[idx]["question"], self.vecAnnos[idx]["answerable"], self.vecAnnos[idx]["answers"]

In [ ]:
# Path to images
train_dir = r"C:\Users\Jalynn\CSCI5919_LabAssignment3\train\train"
val_dir = r"C:\Users\Jalynn\CSCI5919_LabAssignment3\val\val"
test_dir = r"C:\Users\Jalynn\CSCI5919_LabAssignment3\test\test"

# Path to annotation files
train_anno = "annotations/train.json"
val_anno = "annotations/val.json"
test_anno = "annotations/test.json"

In [ ]:
# def __init__(self, strFolder: str, strAnnotationPath: str, fDataPercentage: float = 1.0,

# Training set at 50%
train_dataset = VizWizLoader(train_dir, train_anno, 0.4873)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# "A strict 50% training data limit is enforced"
print(f"The length should be 10k: {len(train_dataset)}")

# # Training set at 75%
# train_dataset = VizWizLoader(train_dir, train_anno, 0.75)
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# print(f"The length out of curiousity: {len(train_dataset)}")

# # Training set at 100%
# train_dataset = VizWizLoader(train_dir, train_anno, 1.0)
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# print(f"The length out of curiousity: {len(train_dataset)}")

# Validation set
val_dataset = VizWizLoader(val_dir, val_anno, 1.0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Test set
test_dataset = VizWizLoader(test_dir, test_anno, 1.0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Pull one batch from the training loader
images, questions, labels, answers = next(iter(train_loader))

print(f"Image batch shape: {images.shape}") # Should be [32, 3, 224, 224]
print(f"First question: {questions[0]}") # The text question
print(f"First label: {labels[0]}") # 0 or 1

## Text Tokenization

In [ ]:
# Lowercase and remove punctuation
def basic_tokenizer(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()

# Build Vocabulary from 10k samples
word_counts = Counter()
for i in range(len(train_dataset)):
    _, question, _, _ = train_dataset[i]
    word_counts.update(basic_tokenizer(question))

# Create word-to-index mapping (reserve 0 for padding, 1 for unknown)
vocab = {word: i+2 for i, (word, _) in enumerate(word_counts.most_common(5000))}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

# Task: Mention design choice in overleaf - 20 limit
def encode_text(text, max_len=20):
    tokens = basic_tokenizer(text)
    encoded = [vocab.get(token, 1) for token in tokens]
    # Adding padding
    if len(encoded) < max_len:
        encoded += [0] * (max_len - len(encoded))
    # Truncate
    else:
        encoded = encoded[:max_len]
    return torch.tensor(encoded)

# Number of unique words -- 2697
# Remaining words treated as unknown
print(f"Vocab size: {len(vocab)}")
# Previous batch question
# Outputs tensor - non zero are ID #s for corresponding words
# Zeros are the padding
print(f"Example encoding: {encode_text('Is my Bose turned on?')}")

In [ ]:
# Review labels - 0s and 1s for answerable or not
print(f"Label type: {labels.dtype}, Label values: {labels[:5]}")

# End-To-End Design: Custom Architecture

In [ ]:
class CustomVQAModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_heads=4, activation_type="ReLU"):
        super(CustomVQAModel, self).__init__()
        
        # Helper to pick the activation
        if activation_type == "GELU":
            self.act = nn.GELU()
        elif activation_type == "LeakyReLU":
            self.act = nn.LeakyReLU()
        else:
            self.act = nn.ReLU()
            
        # 1. Image Branch (Basic CNN)
        self.image_feat = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            self.act,
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            self.act,
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        
        # 2. Text Branch (Self-Attention)
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.text_flatten = nn.Linear(embed_dim, 32)
        
        # 3. Fusion and Classifier
        self.classifier = nn.Sequential(
            nn.Linear(32 + 32, 64),
            self.act,
            nn.Linear(64, 1)
        )

    def forward(self, images, questions):
        img_v = self.image_feat(images)
        text_v = self.embedding(questions)
        text_v = self.transformer(text_v)
        text_v = text_v.mean(dim=1)
        text_v = self.text_flatten(text_v)
        combined = torch.cat((img_v, text_v), dim=1)
        logits = self.classifier(combined)
        return logits.squeeze()

# Challenge #1

In [ ]:
# Experiment list of activation funcs
activations_to_test = ["ReLU", "GELU", "LeakyReLU"]
experiment_results = {}

for act_name in activations_to_test:
    print(f"Starting: {act_name}")
    
    # Uses Task 1 Model
    model_exp = CustomVQAModel(len(vocab), activation_type=act_name).to(device)
    optimizer = optim.Adam(model_exp.parameters(), lr=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    
    # Training loop
    for epoch in range(5):
        model_exp.train()
        loop = tqdm(train_loader, total=len(train_loader), desc=f"{act_name} Epoch {epoch+1}")
        for batch in loop:
            imgs, questions_raw, labels, _ = batch
            questions_encoded = torch.stack([encode_text(q) for q in questions_raw]).to(device)
            imgs, labels = imgs.to(device), labels.to(device).float()
            
            optimizer.zero_grad()
            outputs = model_exp(imgs, questions_encoded)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            loop.set_postfix(loss=loss.item())

    # Validation loop
    model_exp.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            imgs, questions_raw, labels, _ = batch
            questions_encoded = torch.stack([encode_text(q) for q in questions_raw]).to(device)
            imgs, labels = imgs.to(device), labels.to(device)
            
            outputs = model_exp(imgs, questions_encoded)
            predictions = (torch.sigmoid(outputs) > 0.5).int()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    
    acc = 100 * correct / total
    experiment_results[act_name] = acc
    print(f"Done: {act_name} achieved {acc:.2f}% Accuracy")

    # Clean up GPU memory
    del model_exp
    torch.cuda.empty_cache()

# final comparison for report
print("FINAL RESULTS TABLE")
for name, accuracy in experiment_results.items():
    print(f"{name:<12} | Accuracy: {accuracy:.2f}%")

In [ ]:
winner_act = "LeakyRELU"

In [ ]:
best_model = CustomVQAModel(len(vocab), activation_type=winner_act).to(device)
optimizer = optim.Adam(best_model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

# 5-epoch train for the final submission
for epoch in range(5):
    best_model.train()
    for batch in train_loader:
        imgs, questions_raw, labels, _ = batch
        questions_encoded = torch.stack([encode_text(q) for q in questions_raw]).to(device)
        imgs, labels = imgs.to(device), labels.to(device).float()
        optimizer.zero_grad()
        outputs = best_model(imgs, questions_encoded)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

## Challenge #1 File Export

In [ ]:
best_model.eval()
test_predictions = []

with torch.no_grad():
    for i in range(100, 200):
        img, question = test_dataset[i]
        img = img.unsqueeze(0).to(device)
        question_enc = encode_text(question).unsqueeze(0).to(device)
        output = best_model(img, question_enc)
        pred = (torch.sigmoid(output) > 0.5).int()
        test_predictions.append(pred.item())

submission_tensor = torch.tensor(test_predictions)
filename = "jalynn_nicoly_challenge1.pkl"
# Use torch.save()
torch.save(submission_tensor, filename)

print(f"Saved 100 predictions using {winner_act} to {filename}")
print(f"Preview: {submission_tensor}")

# Challenge #2

In [ ]:
# All answers from 10k training samples
all_answers = []

for i in range(10000):
    # Unpack VizWizLoader: (image, question, label, answers)
    _, _, _, raw_answers = train_dataset[i]
    
    # Obtain answer string
    for a in raw_answers:
        if isinstance(a, dict):
            all_answers.append(a['answer'])
        else:
            all_answers.append(a)

# Counter
most_common = Counter(all_answers).most_common(1000)
ans_vocab = {ans[0]: i for i, ans in enumerate(most_common)}
ans_list = [ans[0] for ans in most_common]

print(f"Answer Vocab Size: {len(ans_vocab)}")
print(f"Top 5 Answers: {ans_list[:5]}")

In [ ]:
class Challenge2Model(CustomVQAModel): 
    def __init__(self, vocab_size, num_answers=1000, dropout_p=0.2):
        super().__init__(vocab_size, activation_type="LeakyReLU")
        
        # Here is where the hidden layer size increases
        self.classifier = nn.Sequential(
            nn.Linear(32 + 32, 256),
            nn.LeakyReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(256, num_answers) 
        )

    def forward(self, images, questions):
        img_v = self.image_feat(images)
        text_v = self.embedding(questions)
        text_v = self.transformer(text_v).mean(dim=1)
        text_v = self.text_flatten(text_v)
        
        combined = torch.cat((img_v, text_v), dim=1)
        logits = self.classifier(combined)
        return logits

In [ ]:
def evaluate_human_accuracy(model, loader):
    model.eval()
    total_score = 0
    count = 0
    with torch.no_grad():
        for batch in loader:
            imgs, questions_raw, _, answers_list = batch
            qs = torch.stack([encode_text(q) for q in questions_raw]).to(device)
            outputs = model(imgs.to(device), qs)
            preds = torch.argmax(outputs, dim=1)
            
            batch_size = imgs.size(0)
            for i in range(batch_size):
                pred_idx = preds[i].item()
                pred_text = ans_list[pred_idx]
                
                current_image_answers = []
                for ans_dict in answers_list:
                    ans_str = ans_dict['answer'][i]
                    current_image_answers.append(ans_str)
                
                # Human Accuracy Metric: min(# humans who said it / 3, 1)
                # https://github.com/yousefkotp/Visual-Question-Answering
                num_matches = sum(1 for a in current_image_answers if a == pred_text)
                score = min(num_matches / 3, 1.0)
                total_score += score
                count += 1
                
    return total_score / count

In [ ]:
# List of dropout rates for exp.
dropout_trials = [0.2, 0.5]
trial_results = {}

for p in dropout_trials:
    print(f"Starting: Dropout = {p}")
    
    model_c2 = Challenge2Model(len(vocab), num_answers=1000, dropout_p=p).to(device)
    optimizer = optim.Adam(model_c2.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    # Training set
    for epoch in range(5):
        model_c2.train()
        loop = tqdm(train_loader, desc=f"Dropout {p} - Epoch {epoch+1}")
        for batch in loop:
            imgs, questions_raw, _, answers_list = batch
            qs = torch.stack([encode_text(q) for q in questions_raw]).to(device)
            
            targets = []
            batch_size = imgs.size(0)
            
            for i in range(batch_size):
                current_image_answers = []
                # Loop through the 10 answer dictionaries provided for the batch
                for ans_dict in answers_list:
                    # ans_dict['answer'] is a list of strings for the whole batch
                    ans_str = ans_dict['answer'][i]
                    current_image_answers.append(ans_str)
                
                # Majority vote! from the 10 collected strings
                most_common_text = Counter(current_image_answers).most_common(1)[0][0]
                targets.append(ans_vocab.get(most_common_text, 0))

            targets = torch.tensor(targets).to(device)

            optimizer.zero_grad()
            outputs = model_c2(imgs.to(device), qs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            loop.set_postfix(loss=loss.item())

    # Validation set
    print(f"Calculating Human Accuracy for Dropout {p}...")
    accuracy = evaluate_human_accuracy(model_c2, val_loader)
    trial_results[p] = accuracy
    print(f"Trial Result: {accuracy:.4f}")

# Final printout for your report
print("\nTask Two Results")
for p, acc in trial_results.items():
    print(f"Dropout {p}: {acc:.4f} Human Accuracy")

## Challenge #2 File Export

In [ ]:
model_c2.eval()
submission_list = []

with torch.no_grad():
    for i in range(100, 200):
        image_filename = test_dataset.vecAnnos[i]["image"]
        
        img, question_str = test_dataset[i]
        img_tensor = img.unsqueeze(0).to(device)
        qs_tensor = encode_text(question_str).unsqueeze(0).to(device)
        
        output = model_c2(img_tensor, qs_tensor)
        pred_idx = torch.argmax(output, dim=1).item()
        pred_answer = ans_list[pred_idx]
        
        # JSON
        submission_list.append({
            "image": image_filename,
            "answer": pred_answer
        })

file_name = "jalynn_nicoly_challenge2.json"
with open(file_name, 'w') as f:
    json.dump(submission_list, f, indent=4)

print(f"Successfully saved {len(submission_list)} predictions to {file_name}")
print(submission_list[0])

# Foundation Model Adaptation: CLIP

## Load CLIP Files

In [ ]:
# Load CLIP Files
train_img_feats = torch.load('VizWiz_CLIP/VizWiz_train_CLIP_Image.pkl')
train_txt_feats = torch.load('VizWiz_CLIP/VizWiz_train_CLIP_Text.pkl')
val_img_feats = torch.load('VizWiz_CLIP/VizWiz_val_CLIP_Image.pkl')
val_txt_feats = torch.load('VizWiz_CLIP/VizWiz_val_CLIP_Text.pkl')

# Variable used in challenge 3 and 4
N_LIMIT = 10000
train_img_feats = train_img_feats[:N_LIMIT]
train_txt_feats = train_txt_feats[:N_LIMIT]

# Extract labels as 0 or 1
y_train = torch.tensor([train_dataset.vecAnnos[i]['answerable'] for i in range(N_LIMIT)])
# y_train = torch.tensor([train_dataset.vecAnnos[i]['answerable'] for i in range(len(train_dataset))])
y_val = torch.tensor([val_dataset.vecAnnos[i]['answerable'] for i in range(len(val_dataset))])

print(f"Features loaded. Training on {train_img_feats.shape[0]} samples.")

# Challenge #3

## CLIP Model

In [ ]:
class Challenge3Model(nn.Module):
    def __init__(self, architecture_type="deep"):
        super(Challenge3Model, self).__init__()
        # Shallow
        if architecture_type == "shallow":
            self.net = nn.Sequential(
                nn.Linear(1024, 512),
                nn.ReLU(),
                nn.Linear(512, 1)
            )
        # Deep
        else:
            self.net = nn.Sequential(
                nn.Linear(1024, 512),
                nn.BatchNorm1d(512),
                nn.ReLU(),
                nn.Linear(512, 128),
                nn.ReLU(),
                nn.Linear(128, 1)
            )

    def forward(self, img_feat, txt_feat):
        x = torch.cat((img_feat, txt_feat), dim=1)
        return self.net(x).squeeze()

## Train Model

In [ ]:
experimental_results = {}
conditions = ["shallow", "deep"]

for arch in conditions:
    print(f"\n--- Training Challenge 3: {arch.upper()} Architecture ---")
    model_c3 = Challenge3Model(architecture_type=arch).to(device)
    optimizer = torch.optim.Adam(model_c3.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()
    
    for epoch in range(20):
        model_c3.train()
        optimizer.zero_grad()
        
        outputs = model_c3(train_img_feats.to(device), train_txt_feats.to(device))
        loss = criterion(outputs, y_train.to(device).float())
        
        loss.backward()
        optimizer.step()
        
        # Validation
        model_c3.eval()
        with torch.no_grad():
            val_outputs = model_c3(val_img_feats.to(device), val_txt_feats.to(device))
            val_preds = (torch.sigmoid(val_outputs) > 0.5).int()
            acc = (val_preds.cpu() == y_val).float().mean()
            
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: Loss {loss.item():.4f}, Val Acc: {acc:.4f}")
            
    experimental_results[arch] = acc.item()

# Print summary
print("CHALLENGE 3 RESULTS")
for arch, acc in experimental_results.items():
    print(f"Architecture: {arch:<10} | Val Accuracy: {acc*100:.2f}%")

## Challenge #3 File Export

In [ ]:
best_arch = max(experimental_results, key=experimental_results.get)

model_c3 = Challenge3Model(architecture_type=best_arch).to(device)
test_img_feats = torch.load('VizWiz_CLIP/VizWiz_test_CLIP_Image.pkl')
test_txt_feats = torch.load('VizWiz_CLIP/VizWiz_test_CLIP_Text.pkl')

model_c3.eval()
with torch.no_grad():
    # Indices 100 through 199
    t_img = test_img_feats[100:200].to(device)
    t_txt = test_txt_feats[100:200].to(device)
    
    logits = model_c3(t_img, t_txt)
    final_preds = (torch.sigmoid(logits) > 0.5).int()

# Use torch.save()
torch.save(final_preds.cpu(), "jalynn_nicoly_challenge3.pkl")

# Challenge #4

## CLIP Model

In [ ]:
class Challenge4Model(torch.nn.Module):
    # Hidden dim for exp var
    def __init__(self, num_classes=1000, hidden_dim=1024, dropout_p=0.4):
        super(Challenge4Model, self).__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(1024, hidden_dim), 
            torch.nn.BatchNorm1d(hidden_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout_p),
            torch.nn.Linear(hidden_dim, 512),
            torch.nn.ReLU(),
            torch.nn.Linear(512, num_classes)
        )

    def forward(self, img_feat, txt_feat):
        x = torch.cat((img_feat, txt_feat), dim=1)
        return self.net(x)

## Train Model

In [ ]:
y_train_gen = []

# Recall n limit assigned to 10k - challenge 4 irrelvant to data efficiency 5922 only
for i in range(N_LIMIT):
    raw_answers = train_dataset.vecAnnos[i]['answers']
    ans_strings = [a['answer'] for a in raw_answers]
    
    # Majority vote
    most_common = Counter(ans_strings).most_common(1)[0][0]
    y_train_gen.append(ans_vocab.get(most_common, 0))

y_train_gen = torch.tensor(y_train_gen).to(device)

# Validation targets 
y_val_gen = []
for i in range(len(val_dataset)):
    raw_answers = val_dataset.vecAnnos[i]['answers']
    ans_strings = [a['answer'] for a in raw_answers]
    most_common = Counter(ans_strings).most_common(1)[0][0]
    y_val_gen.append(ans_vocab.get(most_common, 0))

y_val_gen = torch.tensor(y_val_gen).to(device)

In [ ]:
# The experimental variable values
widths_to_test = [512, 1024] # Narrow vs. Wide
challenge4_results = {}

for w in widths_to_test:
    model_c4 = Challenge4Model(num_classes=1000, hidden_dim=w).to(device)
    optimizer = torch.optim.Adam(model_c4.parameters(), lr=1e-3)
    criterion = torch.nn.CrossEntropyLoss()

    for epoch in range(20):
        model_c4.train()
        optimizer.zero_grad()
        outputs = model_c4(train_img_feats.to(device), train_txt_feats.to(device))
        loss = criterion(outputs, y_train_gen)
        loss.backward()
        optimizer.step()
        
    model_c4.eval()
    with torch.no_grad():
        val_outputs = model_c4(val_img_feats.to(device), val_txt_feats.to(device))
        preds = torch.argmax(val_outputs, dim=1)
        acc = (preds.cpu() == y_val_gen.cpu()).float().mean()
        print(f"Final Val Accuracy for width {w}: {acc:.4f}")
        challenge4_results[w] = acc.item()

## Challenge #4 File Export

In [ ]:
best_width = max(challenge4_results, key=challenge4_results.get)
best_accuracy = challenge4_results[best_width]
model_c4 = Challenge4Model(num_classes=1000, hidden_dim=best_width).to(device)

submission_gen = []
model_c4.eval()

with torch.no_grad():
    for i in range(100, 200):
        # Unsqueeze(0) to simulate a batch size of 1
        t_img = test_img_feats[i].unsqueeze(0).to(device)
        t_txt = test_txt_feats[i].unsqueeze(0).to(device)
        
        outputs = model_c4(t_img, t_txt)
        pred_idx = torch.argmax(outputs, dim=1).item()
        
        pred_text = ans_list[pred_idx]
        image_filename = test_dataset.vecAnnos[i]["image"]
        
        submission_gen.append({
            "image": image_filename,
            "answer": pred_text
        })

# Save as .json
file_name = "jalynn_nicoly_challenge4.json"
with open(file_name, 'w') as f:
    json.dump(submission_gen, f, indent=4)